# Aula 09.1 - OpenRouter: Gateway Multi-Modelo

## Objetivo
Usar a API HTTP do OpenRouter para comparar modelos disponiveis no catalogo, incluindo opcoes marcadas como gratuitas. O endpoint tambem e compativel com o formato de chat adotado pelo SDK OpenAI.

## Como o OpenRouter funciona
O OpenRouter atua como um gateway: sua aplicacao chama uma API unica e escolhe o identificador do modelo desejado. O gateway encaminha a solicitacao ao provedor correspondente. Isso simplifica a experimentacao, mas nao elimina limites de uso, regras de privacidade nem a necessidade de validar cada resposta.

## Tipos de modelos no catalogo
- **Chat/Instruct:** para conversa, resumo, extracao e geracao de texto.
- **Reasoning:** para problemas que exigem raciocinio mais elaborado, normalmente com maior latencia.
- **Vision/Multimodal:** recebem texto e imagem; verificar modalidades suportadas.
- **Embeddings:** geram vetores para busca semantica; nao respondem como um chat.

Antes de escolher, confira ID, modalidade, context window, preco ou marcacao gratuita, limites e provedor.

## Antes de comecar
1. Crie uma chave em [OpenRouter](https://openrouter.ai/keys).
2. Abra o catalogo e escolha modelos marcados como gratuitos no momento da aula. A disponibilidade e os limites podem mudar.
3. Defina `OPENROUTER_API_KEY` no ambiente.
4. Nao envie a chave ou dados confidenciais ao servico.

In [1]:
%pip install -q -U requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.2.1
[notice] To update, run: C:\Users\maste\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import requests

api_key = os.getenv('OPENROUTER_API_KEY')
assert api_key, 'Defina OPENROUTER_API_KEY no ambiente antes de continuar.'

ENDPOINT = 'https://openrouter.ai/api/v1/chat/completions'
HEADERS = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}

## Parte 1 - Opcoes gratuitas no catalogo
Consulta realizada em 20/09/2026. Modelos marcados como gratuitos podem mudar, sair do catalogo ou ter rate limits. Antes de executar, confirme o ID e a marcacao no [catalogo do OpenRouter](https://openrouter.ai/models).

| Modelo | Uso sugerido | Contexto informado |
|---|---|---:|
| `qwen/qwen3.8-27b:free` | Chat, texto e multimodal | 262.144 |
| `google/gemma-4-31b-it:free` | Texto, imagem e video | 262.144 |
| `cohere/north-mini-code:free` | Tarefas de codigo | 256.000 |
| `nvidia/nemotron-3.5-lightning:free` | Texto com contexto longo | 1.000.000 |
| `nex-agi/nex-n2.5-pro:free` | Texto e imagem | 262.144 |

Os valores de contexto sao informativos; sempre consulte a ficha atual do modelo.

In [3]:
# Opcoes gratuitas consultadas em 20/09/2026. Confirme no catalogo antes da aula.
MODELOS_GRATUITOS = {
    'qwen_chat': 'qwen/qwen3.8-27b:free',
    'gemma_multimodal': 'google/gemma-4-31b-it:free',
    'cohere_codigo': 'cohere/north-mini-code:free',
    'nemotron_contexto_longo': 'nvidia/nemotron-3.5-lightning:free',
    'nex_multimodal': 'nex-agi/nex-n2.5-pro:free',
}

# Validado nesta aula: Cohere North Mini Code. Outros modelos gratuitos podem retornar 429.
MODELO_ESCOLHIDO = MODELOS_GRATUITOS['cohere_codigo']
prompt = 'Explique RAG em duas frases para uma pessoa desenvolvedora.'
print(MODELO_ESCOLHIDO)

cohere/north-mini-code:free


In [4]:
def perguntar(modelo, texto):
    resposta = requests.post(
        ENDPOINT,
        headers=HEADERS,
        json={
            'model': modelo,
            'messages': [{'role': 'user', 'content': texto}],
        },
        timeout=60,
    )
    if resposta.status_code == 429:
        raise RuntimeError('Rate limit do modelo gratuito. Escolha outra opcao do catalogo e tente novamente.')
    resposta.raise_for_status()
    return resposta.json()['choices'][0]['message']['content']

resposta = perguntar(MODELO_ESCOLHIDO, prompt)
print(resposta)

RAG (Retrieval‑Augmented Generation) combina um sistema de busca que recupera informações relevantes de um conjunto de dados externo com um modelo de linguagem gerativo, permitindo que o modelo produza respostas precisas e baseadas em fatos ao consultar um grande corpus. Para os desenvolvedores, é uma maneira de aprimorar os LLMs com conhecimento atualizado ou específico do domínio sem exigir que o modelo memorize todos os detalhes, garantindo assim outputs mais confiáveis e referenciados.


## Parte 2 - Comparacao
Para cada modelo, avalie qualidade, tempo de resposta, aderencia ao prompt e erros. Registre tambem se o modelo continua marcado como gratuito e se houve rate limit.

In [5]:
# TODO: preencha com a analise da dupla.
comparativo = {
    'modelo': [],
    'qualidade': [],
    'latencia_percebida': [],
    'limites_observados': [],
}
comparativo

{'modelo': [],
 'qualidade': [],
 'latencia_percebida': [],
 'limites_observados': []}

## Reflexao
- Vantagem: uma integracao para experimentar varios modelos e provedores.
- Limite: modelos gratuitos, capacidade e regras podem variar.
- Cuidado: avalie quais dados podem passar pelo gateway e pelo provedor selecionado.

## Entrega
Notebook executado, tabela comparativa e recomendacao justificada de modelo para um caso de uso.